In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Imports & Global Settings

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

Define Agronomic Constants

In [2]:
# Baseline annual NPK requirements (kg/ha/year)
BASE_NPK = {
    "Arabica": {"N": 120, "P": 40, "K": 120},
    "Robusta": {"N": 150, "P": 50, "K": 150}
}

# Reference yield (kg/ha) for scaling
REFERENCE_YIELD = {
    "Arabica": 1500,
    "Robusta": 2000
}

# Target soil nutrient levels (mg/kg - relative)
TARGET_SOIL = {"N": 50, "P": 30, "K": 120}

Helper Functions

In [4]:
def yield_factor(pred_yield, variety):
    return np.clip(pred_yield / REFERENCE_YIELD[variety], 0.7, 1.3)


def soil_factor(measured, target):
    return np.clip(target / measured, 0.8, 1.2)


def rainfall_factor(rain_mm):
    if rain_mm > 200:
        return 1.1
    elif rain_mm < 80:
        return 0.95
    return 1.0


def growth_stage_factor(stage):
    if stage == "vegetative":
        return {"N": 1.15, "P": 1.0, "K": 1.0}
    if stage == "flowering":
        return {"N": 1.0, "P": 1.15, "K": 1.0}
    if stage == "fruiting":
        return {"N": 1.0, "P": 1.0, "K": 1.2}
    return {"N": 0.9, "P": 0.9, "K": 0.9}

Generate Synthetic Dataset

In [5]:
rows = []

for year in range(2023, 2027):
    for month in range(1, 13):
        for variety in ["Arabica", "Robusta"]:

            predicted_yield = np.random.uniform(
                REFERENCE_YIELD[variety] * 0.7,
                REFERENCE_YIELD[variety] * 1.3
            )

            soil_n = np.random.uniform(30, 70)
            soil_p = np.random.uniform(15, 45)
            soil_k = np.random.uniform(80, 160)

            rainfall = np.random.uniform(60, 300)
            temperature = np.random.uniform(18, 28)

            growth_stage = np.random.choice(
                ["vegetative", "flowering", "fruiting", "post-harvest"]
            )

            base = BASE_NPK[variety]

            yf = yield_factor(predicted_yield, variety)
            rf = rainfall_factor(rainfall)
            sf = growth_stage_factor(growth_stage)

            N = base["N"] * yf * rf * sf["N"] * soil_factor(soil_n, TARGET_SOIL["N"])
            P = base["P"] * yf * sf["P"] * soil_factor(soil_p, TARGET_SOIL["P"])
            K = base["K"] * yf * rf * sf["K"] * soil_factor(soil_k, TARGET_SOIL["K"])

            # Add noise
            N *= np.random.uniform(0.95, 1.05)
            P *= np.random.uniform(0.95, 1.05)
            K *= np.random.uniform(0.95, 1.05)

            rows.append({
                "year": year,
                "month": month,
                "coffee_variety": variety,
                "predicted_yield_kg_per_ha": round(predicted_yield, 1),
                "soil_n": round(soil_n, 1),
                "soil_p": round(soil_p, 1),
                "soil_k": round(soil_k, 1),
                "rainfall_mm": round(rainfall, 1),
                "avg_temperature_c": round(temperature, 1),
                "growth_stage": growth_stage,
                "N_kg_per_ha": round(N, 1),
                "P_kg_per_ha": round(P, 1),
                "K_kg_per_ha": round(K, 1)
            })

Create DataFrame & Apply Safety Constraints

In [6]:
df = pd.DataFrame(rows)

df["N_kg_per_ha"] = df["N_kg_per_ha"].clip(50, 250)
df["P_kg_per_ha"] = df["P_kg_per_ha"].clip(20, 100)
df["K_kg_per_ha"] = df["K_kg_per_ha"].clip(50, 250)

df.head()

,year,month,coffee_variety,predicted_yield_kg_per_ha,soil_n,soil_p,soil_k,rainfall_mm,avg_temperature_c,growth_stage,N_kg_per_ha,P_kg_per_ha,K_kg_per_ha
0,2023,1,Arabica,1387.1,68.0,37.0,127.9,97.4,19.6,fruiting,88.4,29.5,120.5
1,2023,1,Robusta,2181.1,32.3,36.7,155.1,60.2,27.9,vegetative,210.3,44.7,123.5
2,2023,2,Arabica,1312.1,54.5,19.2,103.4,147.9,22.6,fruiting,97.5,41.5,153.3
3,2023,2,Robusta,1960.1,64.4,35.4,116.0,63.2,27.4,flowering,115.2,46.8,138.6
4,2023,3,Arabica,1665.8,47.6,18.7,119.6,68.3,27.1,post-harvest,115.9,49.2,113.5


Save Dataset

In [7]:
df.to_csv(
    "/content/drive/MyDrive/Coffee/Dev/C3/data/coffee_fertilizer_dataset_per_ha.csv",
    index=False
)

print("Fertilizer dataset saved successfully")

Fertilizer dataset saved successfully
